# Complete fluorescence-array addressing: one 10 × 10 example

Run the cells from top to bottom, or choose **Runtime → Run all**. This notebook installs the environment, downloads the paper YOLO weights, loads ten selected raw frames from sequence **N008**, and displays each processing stage. CPU is supported; a GPU is optional.

**Raw frames → temporal maximum projection → background subtraction and contrast enhancement → YOLO boxes → candidate filtering → joint image–geometry registration → completed 10 × 10 address grid.**

The ten full-field frames are a lossless, preselected subset of one 60-frame acquisition. Enhancement runs on the full field before the fixed display/analysis crop. No manual center coordinates are loaded during inference.


In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY_URL = 'https://github.com/ShuoYang-buct/convective-pcr-array-addressing.git'
ROOT = Path('/content/pcr-array-demo')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(ROOT)], check=True)
else:
    origin = subprocess.check_output(['git', '-C', str(ROOT), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin != REPOSITORY_URL:
        raise ValueError('The existing folder belongs to another project; restart the runtime.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements.txt')], check=True)
sys.path.insert(0, str(ROOT/'src'))
print('Environment ready.')


## 1. Download the YOLO model
The weight file is provided as a GitHub Release asset. Its SHA-256 must match the paper model before inference starts.


In [ ]:
import json, hashlib, urllib.request
manifest = json.loads((ROOT/'model_manifest.json').read_text())
WEIGHTS = ROOT/'weights/best.pt'
WEIGHTS.parent.mkdir(exist_ok=True)
if not WEIGHTS.exists() or hashlib.sha256(WEIGHTS.read_bytes()).hexdigest() != manifest['sha256']:
    temporary = WEIGHTS.with_suffix('.download')
    urllib.request.urlretrieve(manifest['public_url'], temporary)
    if hashlib.sha256(temporary.read_bytes()).hexdigest() != manifest['sha256']:
        raise ValueError('Model weight checksum mismatch.')
    temporary.replace(WEIGHTS)
print('Paper model verified:', manifest['architecture'])


## 2. Load one group of original images
N008 contains a 10 × 10 array. The filenames below identify the ten selected raw frames; their original pixel values are preserved in PNG format. Only the fixed crop is used for display here.


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from demo_visuals import show_stage, completion_box_size
META = json.loads((ROOT/'demo/N008/metadata.json').read_text())
frames = []
for name in META['frames']:
    path = ROOT/'demo/N008'/name
    if hashlib.sha256(path.read_bytes()).hexdigest() != META['frame_sha256'][name]:
        raise ValueError('Demo image checksum mismatch.')
    image = cv2.imdecode(np.fromfile(path, np.uint8), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Cannot read {name}')
    frames.append(image)
x, y, width, height = META['crop_xywh']
crop = lambda image: image[y:y+height, x:x+width]
OUT = ROOT/'outputs/demo'
OUT.mkdir(parents=True, exist_ok=True)
print('Sequence:', META['sequence'], '| layout:', META['rows'], '×', META['cols'], '| selected frames:', len(frames))
display(show_stage(crop(frames[0]), 'Original acquired frame (display crop)', save=OUT/'01_source_frame.png'))
plt.close('all')


## 3. Temporal aggregation
Take the pixelwise maximum across the ten selected frames. This allows a site that becomes brighter at a different time to contribute to the reference image.


In [ ]:
projection = np.maximum.reduce(frames)
display(show_stage(crop(projection), 'Maximum projection of 10 selected frames', save=OUT/'02_maximum_projection.png'))
plt.close('all')


## 4. Background subtraction and contrast enhancement
Apply the paper pipeline on the full-field projection: 3 × 3 median filtering, Gaussian background subtraction (σ = 60 pixels), percentile stretching (1–99.5%), and CLAHE (clip limit 2.0; 64 × 64 tiles). Crop after enhancement so that local contrast processing matches the paper reference.


In [ ]:
from preprocessing import enhance_pipeline1_basic
reference = crop(enhance_pipeline1_basic(projection))
display(show_stage(reference, 'Enhanced reference', save=OUT/'03_enhanced_reference.png'))
plt.close('all')


## 5. YOLO detection
Run the trained model now; this cell does not replay saved detection boxes. Cyan rectangles show YOLO boxes at confidence ≥ 0.25, before the paper's candidate filtering.


In [ ]:
from detector import predict_yolo_boxes
raw_boxes = predict_yolo_boxes(reference, WEIGHTS)
print('YOLO boxes:', len(raw_boxes))
display(show_stage(reference, f'YOLO detection: {len(raw_boxes)} boxes', boxes=raw_boxes, save=OUT/'04_yolo_boxes.png'))
plt.close('all')


## 6. Candidate filtering
Retain confidence ≥ 0.5, remove small and overlapping boxes and isolated candidates, and normalize box sizes using the median detector width and height. The same filtering/normalization sequence is used by the paper's inference code.


In [ ]:
from detector import prepare_candidates
candidates = prepare_candidates(raw_boxes, reference.shape)
print('Filtered candidates:', len(candidates))
display(show_stage(reference, f'Filtered candidates: {len(candidates)}', boxes=candidates, save=OUT/'05_filtered_candidates.png'))
plt.close('all')


## 7. Joint image–geometry registration and completion
Estimate array orientation, spacing and address origin using the candidate positions together with local image evidence. The specified layout is 10 × 10. Green boxes mark grid addresses supported by matched candidates; orange boxes mark inferred addresses without matched candidates. Uniform box dimensions use the median candidate width and height.

A completed grid contains all 100 addresses; this does not mean YOLO directly detected every site. QC status and reasons are printed separately.


In [ ]:
from lattice_registration import estimate_lattice
result = estimate_lattice(candidates, META['rows'], META['cols'], image=reference)
if not result.get('ok'):
    raise RuntimeError(result)
grid = np.asarray(result['grid'])
supported = {int(index) for index in result['pairs']}
summary = {
    'sequence': META['sequence'], 'layout': [META['rows'], META['cols']],
    'yolo_boxes': len(raw_boxes), 'filtered_candidates': len(candidates),
    'output_addresses': len(grid), 'candidate_supported_addresses': len(supported),
    'completed_addresses': len(grid)-len(supported),
    'qc_status': result['status'], 'qc_reasons': result['review_reasons']
}
display(summary)
display(show_stage(reference, f"Complete 10 × 10 address grid: {len(grid)} positions", grid=grid, supported=supported,
                   box_size=completion_box_size(candidates), save=OUT/'06_complete_grid.png'))
plt.close('all')


## 8. Save the final image and address table
The download contains all stage images, the final 100-address overlay, predicted row/column coordinates and the QC summary.


In [ ]:
import csv, shutil
(OUT/'prediction.json').write_text(json.dumps(result, indent=2))
(OUT/'summary.json').write_text(json.dumps(summary, indent=2))
with (OUT/'addresses.csv').open('w', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['row', 'column', 'center_x', 'center_y', 'candidate_supported'])
    for index, (gx, gy) in enumerate(grid):
        writer.writerow([index//META['cols']+1, index%META['cols']+1, gx, gy, index in supported])
archive = shutil.make_archive(str(ROOT/'outputs/N008_demo_results'), 'zip', OUT)
from google.colab import files
files.download(archive)
